In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import os
print(os.listdir('/content/drive/MyDrive/Colab Notebooks'))

['Ders__25_GRU (Gated Recurrent Unit). Onlarin sentiment analizi meselesinde tetbiqine numune.ipynb', 'Untitled0.ipynb', 'Untitled (4)', 'Untitled1.ipynb', 'Untitled (3)', 'week9_class_Deep_Learning.ipynb', 'week9_case_study_Deep_Learning_Colab.ipynb', 'wekk9_homework_Deep_Learning_Colab.ipynb', 'time_series.ipynb', 'Prophet.ipynb', 'Untitled2.ipynb', 'Untitled3.ipynb', 'week11_class_CNN.ipynb', 'Untitled4.ipynb', 'Untitled6.ipynb', 'CNN_Use_Ready_Model_EfficientNet.ipynb', 'week11_homework_CNN_Use_Ready_Model_Resnet18.ipynb', 'Untitled5.ipynb', 'week11_homework_CNN_Use_Ready_Model_inception_v4.ipynb', 'week12_class_Fastai_Tabular_Vision.ipynb', 'Untitled7.ipynb', 'week12_case_study_Fastai_Vision_Multi_Class.ipynb', 'Untitled (2)', 'week12_homework_Fastai_Multi_Class.ipynb', 'Untitled8.ipynb', 'Ders_7_Keras – Installation and First Introduction.ipynb', 'Ders_8_Keras - Neyron sebekesinin elyazma reqemlerini tanimaq ucun oyredilmesi.ipynb', 'Ders_9_Neyron shebeke reqemleri necə taniyir.i

In [25]:
import os
print('train_data_true.txt' in os.listdir('/content/drive/MyDrive/Colab Notebooks'))
print('train_data_false.txt' in os.listdir('/content/drive/MyDrive/Colab Notebooks'))

True
True


In [26]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import re

from tensorflow.keras.layers import Dense, GRU, Input, Dropout, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences

with open('/content/drive/MyDrive/Colab Notebooks/train_data_true.txt', 'r', encoding='utf-8') as f:
    texts_true = f.readlines()
    texts_true[0] = texts_true[0].replace('\ufeff', '') #убираем первый невидимый символ

with open('/content/drive/MyDrive/Colab Notebooks/train_data_false.txt', 'r', encoding='utf-8') as f:
    texts_false = f.readlines()
    texts_false[0] = texts_false[0].replace('\ufeff', '') #убираем первый невидимый символ

texts = texts_true + texts_false
count_true = len(texts_true)
count_false = len(texts_false)
total_lines = count_true + count_false
print(count_true, count_false, total_lines)


maxWordsCount = 1000
tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»', lower=True, split=' ', char_level=False)
tokenizer.fit_on_texts(texts)

dist = list(tokenizer.word_counts.items())
print(dist[:10])
print(texts[0][:100])


max_text_len = 10
data = tokenizer.texts_to_sequences(texts)
data_pad = pad_sequences(data, maxlen=max_text_len)
print(data_pad)

print( list(tokenizer.word_index.items()) )


X = data_pad
Y = np.array([[1, 0]]*count_true + [[0, 1]]*count_false)
print(X.shape, Y.shape)

indeces = np.random.choice(X.shape[0], size=X.shape[0], replace=False)
X = X[indeces]
Y = Y[indeces]


model = Sequential()
model.add(Embedding(maxWordsCount, 128, input_length = max_text_len))
model.add(GRU(128, return_sequences=True))
model.add(GRU(64))
model.add(Dense(2, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer=Adam(0.0001))

history = model.fit(X, Y, batch_size=32, epochs=50)

85 88 173
[('думайте', 1), ('позитивно', 4), ('и', 50), ('верьте', 3), ('в', 38), ('свою', 4), ('способность', 1), ('достигать', 1), ('отличных', 1), ('результатов', 1)]
Думайте позитивно и верьте в свою способность достигать отличных результатов. 

[[197  54   2 ... 199 200 201]
 [  0   4 202 ... 205   3  67]
 [206   3  67 ...   4 208 209]
 ...
 [  0  20  62 ...  53 853 854]
 [  0   0  43 ...  33   1 856]
 [  0   0   0 ...  70  65 194]]
[('не', 1), ('и', 2), ('в', 3), ('вы', 4), ('на', 5), ('я', 6), ('а', 7), ('только', 8), ('что', 9), ('все', 10), ('это', 11), ('если', 12), ('жизнь', 13), ('просто', 14), ('с', 15), ('чем', 16), ('никогда', 17), ('чтобы', 18), ('к', 19), ('у', 20), ('больше', 21), ('себя', 22), ('то', 23), ('всегда', 24), ('вас', 25), ('меня', 26), ('за', 27), ('вам', 28), ('когда', 29), ('от', 30), ('мне', 31), ('быть', 32), ('жизни', 33), ('есть', 34), ('но', 35), ('сделать', 36), ('без', 37), ('позитивным', 38), ('никому', 39), ('как', 40), ('ты', 41), ('тем', 42),

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.5665 - loss: 0.6921
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.6532 - loss: 0.6877
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7225 - loss: 0.6831
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7457 - loss: 0.6782
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7572 - loss: 0.6730
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7457 - loss: 0.6658
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7746 - loss: 0.6577
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8035 - loss: 0.6470
Epoch 9/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8266 - loss: 0.6335
Epoch 10/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8728 - loss: 0.6149
Epoch 11/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9133 - loss: 0.5918
Epoch 12/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9191 - loss: 0.5647
E

In [27]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 10, 128)        │       128,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ (None, 10, 128)        │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 793,352 (3.03 MB)

 Trainable params: 264,450 (1.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 528,902 (2.02 MB)

In [31]:
reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))

def sequence_to_text(list_of_indices):
    words = [reverse_word_map.get(letter) for letter in list_of_indices]
    return(words)

t = "Это шедевр изобразительного".lower()
data = tokenizer.texts_to_sequences([t])
data_pad = pad_sequences(data, maxlen=max_text_len)
print( sequence_to_text(data[0]) )

res = model.predict(data_pad)
print(res, np.argmax(res), sep='\n')

['это', 'шедевр', 'изобразительного']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
[[0.7087665  0.29123348]]
0
